Modeling: Chaining
==================

Non-linear search chaining is an advanced model-fitting approach which breaks the model-fitting procedure down into
multiple non-linear searches, using the results of the initial searches to initialization parameter
sampling in subsequent searches. This contrasts the `modeling` examples which each compose and fit a single lens
model-fit using one non-linear search.

The benefits of non-linear search chaining are:

 - Earlier searches fit simpler lens models than the later searches, which have a less complex non-linear parameter
 space that can be sampled more efficiently, with a reduced chance of inferring an incorrect local maxima solution.

 - Earlier searches can use faster non-linear search settings which infer the highest log likelihood models but not
 precisely quantify the parameter errors, with only the final searches using slow settings to robustly estimate errors.

 - Earlier searches can augment the data or alter the fitting-procedure in ways that speed up the computational run
 time. These may impact the quality of the model-fit overall, but they can be reverted to the more accurate but more
 computationally expense setting in the final searches.

__Contents__

- **Concise Model Composition API:** Chaining uses the concise `Model` API to compose lens models, which is nearly identical to the standard API but avoids the need to use `Model` objects to compose the lens model when a light or mass profile is passed to a `Collection` object.
- **This Example:** This script gives an overview of the API for search chaining, a description of how the priors on parameters are used to pass information between searches as well as tools for customizing prior passing.
- **Dataset + Masking:** Load, plot and mask the `Imaging` data.
- **Paths:** The path the results of all chained searches are output.
- **Model (Search 1):** We compose our lens model using `Model` objects, which represent the galaxies we fit to our data.
- **Search + Analysis + Model-Fit (Search 1):** We now create the non-linear search, analysis and perform the model-fit using this model.
- **Result (Search 1):** The results which are used for prior passing are summarised in the `info` attribute.
- **Model Chaining:** We use the results of search 1 to create the `Model` components that we fit in search 2.
- **Model Centred Chaining:** We use the results of search 1 to create the `Model` components that we fit in search 2, using `model_centred` to pass priors centered on the maximum likelihood parameter values.
- **Search + Analysis + Model-Fit (Search 2):** We now create the non-linear search, analysis and perform the model-fit using the chained model.
- **Result (Search 2):** The final results of the chained model-fit.
- **How is Search Chaining Used?:** Overview of how search chaining is used in practice for automated lens modeling and SLaM pipelines.
- **Detailed Explanation Of Prior Passing:** Detailed overview of how prior passing works and tools to customize its behaviour.
- **EXAMPLE:** Lets go through an example using a real parameter.

__Concise Model Composition API__

Chaining uses the concise `Model` API to compose lens models, which is nearly identical to
the standard API but avoids the need to use `Model` objects to compose the lens model when a light or mass
profile is passed to a `Collection` object.

__This Example__

This script gives an overview of the API for search chaining, a description of how the priors on parameters are used
to pass information between searches as well as tools for customizing prior passing.

There are examples throughout the workspace where search chaining improves and helps automate lens modeling.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset + Masking__ 

Load, plot and mask the `Imaging` data.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/imaging/simulator.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=dataset.grid,
    sub_size_list=[4, 2, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(0.0, 0.0)],
)

dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

aplt.subplot_imaging_dataset(dataset=dataset)

__Paths__

The path the results of all chained searches are output:

In [ ]:
path_prefix = Path("imaging") / "chaining" / "start_here"

__Model (Search 1)__

We compose our lens model using `Model` objects, which represent the galaxies we fit to our data. In the first
search our lens model is:

 - The lens galaxy's total mass distribution is an `Isothermal` [7 parameters].
 
 - an MGE with 1 x 20 Gaussians for the source galaxy's light [4 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=14.

In [ ]:
lens = af.Model(al.Galaxy, redshift=0.5, mass=al.mp.Isothermal)

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

model_1 = af.Collection(galaxies=af.Collection(lens=lens, source=source))

The `info` attribute shows the model in a readable format.

In [ ]:
print(model_1.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(model_1).figure()

__Search + Analysis + Model-Fit (Search 1)__

We now create the non-linear search, analysis and perform the model-fit using this model.

You may wish to inspect the results of the search 1 model-fit to ensure a fast non-linear search has been provided that 
provides a reasonably accurate lens model.

In [ ]:
search_1 = af.Nautilus(
    path_prefix=path_prefix,
    name="search[1]__start_here",
    unique_tag=dataset_name,
    n_live=100,
)

analysis_1 = al.AnalysisImaging(dataset=dataset, use_jax=True)

result_1 = search_1.fit(model=model_1, analysis=analysis_1)

__Result (Search 1)__

The results which are used for prior passing are summarised in the `info` attribute.

In [ ]:
print(result_1.info)

__Model Chaining__

We use the results of search 1 to create the `Model` components that we fit in search 2.

The term `model` below passes the lens and source models as model-components that are to be fitted
for by the non-linear search. 

In [ ]:
model_2 = af.Collection(
    galaxies=af.Collection(
        lens=result_1.model.galaxies.lens, source=result_1.model.galaxies.source
    ),
)

The `info` attribute shows the model, including how parameters and priors were passed from `result_1`.

In [ ]:
print(model_2.info)

Passing `result_1.model` gives back the same model structure with the same configuration priors, which is the 
point of this section: nothing about the model has changed.

In [ ]:
af.ModelPlotter(model_2).figure()

The priors on the model components are the same as their original priors in `model_1`, that is the priors loaded
from the default configuration files. 

This makes model and search chaining pointless, as the second fit has the same initialization to sample parameter 
space as the first.

We instead want the model to be passed in a way where the priors are updated to reflect the inferred high likelihood 
regions of the first in the first model. To do this we use the `model_centred` attribute of the result, which passes 
the model components with priors that are centered on the maximum likelihood parameter values of the first search.

__Model Centred Chaining__

We use the results of search 1 to create the `Model` components that we fit in search 2.

The term `model_centred` below passes the lens and source models as model-components that are to be fitted
for by the non-linear search. In other chaining examples, we'll see other ways to pass prior results.

In [ ]:
model_2 = af.Collection(
    galaxies=af.Collection(
        lens=result_1.model_centred.galaxies.lens,
        source=result_1.model_centred.galaxies.source,
    ),
)

The `info` attribute shows the model, including how parameters and priors were passed from `result_1`.

This now contains `GaussianPrior`'s that are centered on the maximum likelihood parameter values of the first search.

In [ ]:
print(model_2.info)

Prior passing never alters the structure of a model, only the priors behind it: every prior is now a 
`GaussianPrior` centred on the maximum likelihood value of search 1.

In [ ]:
af.ModelPlotter(model_2).figure()

__Search + Analysis + Model-Fit (Search 2)__

We now create the non-linear search, analysis and perform the model-fit using this model.

You may wish to inspect the `model.info` file of the search 2 model-fit to ensure the priors were passed correctly, as 
well as the checkout the results to ensure an accurate power-law mass model is inferred.

In [ ]:
search_2 = af.Nautilus(
    path_prefix=path_prefix,
    name="search[2]__start_here",
    unique_tag=dataset_name,
    n_live=75,
)

analysis_2 = al.AnalysisImaging(dataset=dataset, use_jax=True)

result_2 = search_2.fit(model=model_2, analysis=analysis_2)

__Result (Search 2)__

The final results can be summarised via printing `info`.

In [ ]:
print(result_2.info)

We will expand on this API in the following tutorials. The main thing to note is that we can pass entire profiles or
galaxies using prior passing, if their model does not change. 

The API to pass a whole profile or galaxy is as follows:
 
 bulge = result_1.model_centred.galaxies.lens.bulge
 lens = result_1.model_centred.galaxies.lens
 source = result_1.model_centred.galaxies.source
 
We can also pass priors using an `instance` instead of a `model_centred`. When an `instance` is used, the maximum likelihood
parameter values are passed as fixed values that are therefore not fitted for by the non-linear search (reducing its
dimensionality). 

We will use this in other examples to "split up" the components of the model we fit, for example fit the lens light, 
and then fix it to the best-fit model in a second search which fits the mass and source.
 
__How is Search Chaining Used?__
 
For a lot of the history of **PyAutoLens***, the passing of priors illustrated in this tutorial has been a key
aspect of how search chaining is used to model strong lenses.

However, since ~2024, the direct passing of priors between searches has become less important. This is because
the non-linear search (Nautilus) is a lot more robust and run times are a lot faster since the inclusion of JAX. Thus,
the need to change priors via `model_centred` has become less important, but can still offer more reliable results
and efficient lens modeling.

In fact, most example chaining examples and the Source, Light and Mass pipeline which are built are this feature
**no longer use prior passing**, always using the `model` attribute to pass models between searches which resets
the prior on parameters to their default priors.

However, search chaining is still integral to automated lens modeling and the SLaM pipelines, but it is now used
to:

- Split up the components of the lens model to be fitted in different searches (e.g. fit lens light and the 
  mass + source in different searches using `instance`).

- Fit simpler models (e.g. MGE source) in earlier searches to get a fast estimate of the lens light and mass models,
  which are then used as fixed components in later searches.
  
- To pass `adapt_images` through the pipelines, which use the results of earlier searches to adapt aspects of the
  lens model to the data in later searches. This is especially important for pixelized sources and described in the 
  `features/pixelization/adaptive` example.

The most robust automated lens modeling is therefore still built using search chaining, but the direct passing of priors
between searches is no longer the main tool by which this is achieved.
 
__Detailed Explanation Of Prior Passing__
 
Lets now think about how priors are passed. Checkout the `model.info` file of the second search of this tutorial. The 
parameters do not use the default priors we saw in search 1 (which are typically broad UniformPriors). Instead, 
they use TruncatedGaussianPrior`s where:

 - The mean values are the median PDF results of every parameter in search 1.
 - The sigma values are specified in the `width_modifier` field of the profile's entry in the `priors.yaml' config 
   file (we will discuss why this is used in a moment).

Like the manual `TruncatedGaussianPrior`'s that were used in tutorial 1, the prior passing API sets up the prior on each 
parameter with a `TruncatedGaussianPrior` centred on the high likelihood regions of parameter space!

To end, I provide a detailed overview of how prior passing works and illustrate tools that can be used to customize
its behaviour. It is up to you whether you want read this, or go ahead to the next tutorial!

Lets say I chain two parameters as follows:

 `mass.einstein_radius = result_1.model_centred.galaxies.lens.mass.einstein_radius`

By invoking the `model_centred` attribute, the prior is passed following 3 rules:

 1) The new parameter, in this case the einstein radius, uses a `TruncatedGaussianPrior`.This is ideal, as the 1D pdf results 
 we compute at the end of a search are easily summarised as a Gaussian.

 2) The mean of the `TruncatedGaussianPrior` is the median PDF value of the parameter estimated in search 1.

 This ensures that the initial sampling of the new search's non-linear starts by searching the region of non-linear 
 parameter space that correspond to highest log likelihood solutions in the previous search. Our priors therefore 
 correspond to the `correct` regions of parameter space.

 3) The sigma of the Gaussian uses the value specified for the profile in the `config/priors/*.yaml` config file's 
 `width_modifer` field (check these files out now).

The idea here is simple. We want a value of sigma that gives a `TruncatedGaussianPrior` wide enough to search a broad 
region of parameter space, so that the lens model can change if a better solution is nearby. However, we want it 
to be narrow enough that we don't search too much of parameter space, as this will be slow or risk leading us 
into an incorrect solution! 

The `width_modifier` values in the priors config file have been chosen based on our experience as being a good
balance broadly sampling parameter space but not being so narrow important solutions are missed.

There are two ways a value is specified using the priors/width file:

 1) Absolute: In this case, the error assumed on the parameter is the value given in the config file. 
 For example, if for the width on centre_0 of a light profile, the width modifier reads "Absolute" with a value 
 0.05. This means if the error on the parameter centre_0 was less than 0.05 in the previous search, the sigma of 
 its `GaussianPrior` in this search will be 0.05.

 2) Relative: In this case, the error assumed on the parameter is the % of the value of the estimated value given in 
 the config file. For example, if the intensity estimated in the previous search was 2.0, and the relative error in 
 the config file reads "Relative" with a value 0.5, then the sigma of the `GaussianPrior` will be 50% of this 
 value, i.e. sigma = 0.5 * 2.0 = 1.0.

We use absolute and relative values for different parameters, depending on their properties. For example, using the 
relative value of a parameter like the `Profile` centre makes no sense. If our lens galaxy is centred at (0.0, 0.0), 
the relative error will always be tiny and thus poorly defined. Therefore, the default configs in **PyAutoLens** use 
absolute errors on the centre.

However, there are parameters where using an absolute value does not make sense. Intensity is a good example of this. 
The intensity of an image depends on its units, S/N, galaxy brightness, etc. There is no single absolute value that 
one can use to generically chain the intensity of any two profiles. Thus, it makes more sense to chain them using 
the relative value from a previous search.

We can customize how priors are passed from the results of a search and non-linear search by editing the
 `prior_passer` settings in the `general.yaml` config file.

__EXAMPLE__

Lets go through an example using a real parameter. Lets say in search 1 we fit the lens galaxy's light with an 
elliptical Sersic profile, and we estimate that its sersic index is equal to 4.0.

To pass this as a prior to search 2 we write:

 lens.bulge.sersic_index = result_1.model_centred.lens.bulge.sersic_index

The prior on the lens galaxy's bulge sersic index in search 2 would thus be a `GaussianPrior` with mean=4.0. 

The value of the Sersic index `width_modifier` in the priors config file sets sigma. The prior config file specifies 
that we use an "Absolute" value of 0.8 to chain this prior. Thus, the `GaussianPrior` in search 2 would have a 
mean=4.0 and sigma=0.8.

If the prior config file had specified that we use an relative value of 0.8, the GaussianPrior in search 2 would have a 
mean=4.0 and sigma = 4.0 * 0.8 = 3.2.

And with that, we're done. Chaining priors is a bit of an art form, but one that works really well. 